# Notebook 04c — Phase 2: MobileFaceNet **Adapter** Fine-tune on Bollywood Faces

**Phase:** 2 (Indian demographic specialization, adapter branch)  
**Purpose:** Sidesteps the ONNX→Keras conversion entirely. Use the existing `mobilefacenet.tflite` (InsightFace baseline) as a **frozen feature extractor**, train a small Keras adapter network on top using ArcFace, and ship the adapter as a separate tiny `.tflite` file. Mobile side runs them sequentially: image → backbone → 512-D → adapter → 512-D Indian-tuned embedding.  
**Expected runtime:** ~40–55 minutes (most of it is the one-time crop + embed pass; actual adapter training is < 5 minutes)  
**GPU required:** No (CPU is fine — embedding pre-compute + small dense head training)

## Why this works when 04a couldn't

We never convert the ONNX MobileFaceNet to a trainable Keras model. The backbone stays as the existing `mobilefacenet.tflite` (frozen, untrainable). We only train an **adapter** — a tiny Keras network (~500k params) that takes the baseline's 512-D output and produces an Indian-face-tuned 512-D output.

## Pipeline

1. **Crop** all ~12k Bollywood images with OpenCV Haar cascade (multi-threaded for speed).
2. **Embed** each crop through `mobilefacenet.tflite` once → save 12k × 512 numpy array. Frozen embeddings — one-time pass.
3. **Build adapter:** input 512-D → Dense + BN + tanh → Dense + BN → residual add → output 512-D. ~500k params.
4. **Train** with ArcFace head (100 Bollywood classes, margin=0.5, scale=64) on the precomputed embeddings. Very fast (no image IO, no big backbone).
5. **EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)** + ModelCheckpoint.
6. **Export** the trained adapter (input `[1, 512]`, output `[1, 512]`) as `mobilefacenet_adapter.tflite`.
7. **Sanity-check** the composed pipeline on a few held-out val celebs.

## What gets shipped to mobile

Two `.tflite` files:

- `mobilefacenet.tflite` — unchanged baseline (13 MB FP32 → ~3.5 MB after INT8 PTQ in Notebook 05)
- `mobilefacenet_adapter.tflite` — the trained adapter (~2 MB FP32 → < 1 MB INT8)

Mobile loader sequence: image → backbone → 512-D raw → adapter → 512-D Indian-tuned → L2-normalize → cosine distance.

## What to send back to Claude

Paste literal output of cells 7, 9, 11, 13, 15, 17, 19, 23, 25, 27 (discovery, gathering, cropping, embedding pre-compute, split, adapter summary, training, TFLite conversion, sanity, final summary). If anything fails, paste the full traceback.

## 1 — Config

In [ ]:
STRATEGY               = "adapter"
NUM_CLASSES            = 100
EPOCHS_MAX             = 30
BATCH_SIZE             = 256
IMAGE_SIZE             = (112, 112)
EMBEDDING_DIM          = 512
LR                     = 1e-3
ARCFACE_SCALE          = 64.0
ARCFACE_MARGIN         = 0.5
VAL_SPLIT              = 0.1
SEED                   = 42
CROP_THREADS           = 2            # reduced from 8 -- cv2.CascadeClassifier is NOT thread-safe across threads. With 2 threads + thread-local classifiers we still get ~2x speedup over sequential without crashing the kernel.
TFLITE_FILENAME        = "mobilefacenet_adapter.tflite"

WORK_DIR    = "/kaggle/working"
MODELS_OUT  = f"{WORK_DIR}/models"
REPORTS_OUT = f"{WORK_DIR}/reports"
CROPS_DIR   = f"{WORK_DIR}/crops"

import os
for d in (MODELS_OUT, REPORTS_OUT, CROPS_DIR):
    os.makedirs(d, exist_ok=True)
print(f"Strategy: {STRATEGY}")
print(f"Config locked. LR={LR}, epochs_max={EPOCHS_MAX}, batch={BATCH_SIZE}, crop_threads={CROP_THREADS}")

## 2 — Clone repo + install

We need `sklearn` only. No ONNX tools, no MediaPipe — this notebook deliberately avoids the install paths that have failed on Kaggle.

In [ ]:
import subprocess

REPO_URL = "https://github.com/MalayM09/nhai.git"
REPO_DIR = os.path.join(WORK_DIR, "nhai")
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

print("\nInstalling deps (sklearn only)…")
!pip install -q scikit-learn

# Verify the baseline mobilefacenet.tflite is in the repo (we use it as the frozen backbone)
BASELINE_TFLITE = os.path.join(REPO_DIR, "mobile_app/assets/models/mobilefacenet.tflite")
assert os.path.isfile(BASELINE_TFLITE), f"baseline TFLite missing: {BASELINE_TFLITE}"
print(f"\nBaseline backbone (frozen): {BASELINE_TFLITE}  ({os.path.getsize(BASELINE_TFLITE)/1024/1024:.2f} MB)")

## 3 — Discover Bollywood dataset

In [ ]:
import glob

print("/kaggle/input contents:")
for d in sorted(os.listdir("/kaggle/input")):
    print(f"  {d}")

DATA_ROOT_CANDIDATES = [
    "/kaggle/input/100-bollywood-celebrity-faces",
    "/kaggle/input/datasets/havingfun/100-bollywood-celebrity-faces",
]
DATA_ROOT = next((c for c in DATA_ROOT_CANDIDATES if os.path.isdir(c)), None)
if DATA_ROOT is None:
    hits = glob.glob("/kaggle/input/**/bollywood_celeb_faces*", recursive=True)
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
assert DATA_ROOT, "Could not locate Bollywood dataset"
print(f"\nDATA_ROOT: {DATA_ROOT}")

SPLIT_FOLDERS = sorted([os.path.join(DATA_ROOT, d) for d in os.listdir(DATA_ROOT)
                       if os.path.isdir(os.path.join(DATA_ROOT, d)) and d.startswith("bollywood_celeb_faces")])
print(f"\nSplit folders ({len(SPLIT_FOLDERS)}):")
for f in SPLIT_FOLDERS:
    n_celebs = len([d for d in os.listdir(f) if os.path.isdir(os.path.join(f, d))])
    print(f"  {f}  ({n_celebs} celebs)")

## 4 — Walk dataset, gather images per celebrity

In [ ]:
from collections import defaultdict

IMG_EXTS = (".jpg", ".jpeg", ".png")
celeb_to_imgs = defaultdict(list)

for split_folder in SPLIT_FOLDERS:
    for celeb_dir in sorted(os.listdir(split_folder)):
        celeb_path = os.path.join(split_folder, celeb_dir)
        if not os.path.isdir(celeb_path):
            continue
        for f in os.listdir(celeb_path):
            if f.lower().endswith(IMG_EXTS):
                celeb_to_imgs[celeb_dir].append(os.path.join(celeb_path, f))

celebs = sorted(celeb_to_imgs.keys())
celeb_to_idx = {c: i for i, c in enumerate(celebs)}
counts = [len(celeb_to_imgs[c]) for c in celebs]
print(f"Total celebs: {len(celebs)}")
print(f"Total images: {sum(counts)}")
print(f"Min/max/median per celeb: {min(counts)}/{max(counts)}/{sorted(counts)[len(counts)//2]}")

if NUM_CLASSES != len(celebs):
    print(f"\nNote: adjusting NUM_CLASSES from {NUM_CLASSES} to {len(celebs)}")
    NUM_CLASSES = len(celebs)

## 5 — Crop faces (multi-threaded for speed)

Same Haar cascade approach as 04b, but using `ThreadPoolExecutor` with 8 workers. cv2.imread + cascade.detectMultiScale release the GIL during native calls, so threading gives a ~5–6× speedup on network-mounted Kaggle datasets. Should cut cropping from ~30 min to ~5–8 min.

In [ ]:
import cv2
import numpy as np
import gc
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
print(f"[1/4] Haar cascade path: {CASCADE_PATH}")

# Verify the cascade loads in the main thread before spawning workers
_test = cv2.CascadeClassifier(CASCADE_PATH)
assert not _test.empty(), "Failed to load Haar cascade in main thread"
del _test
print(f"[2/4] Haar cascade verified")

# Thread-local cascade storage — cv2.CascadeClassifier is NOT thread-safe.
# Each worker thread builds and owns its own classifier.
_thread_local = threading.local()
def get_cascade():
    if not hasattr(_thread_local, "cascade"):
        _thread_local.cascade = cv2.CascadeClassifier(CASCADE_PATH)
    return _thread_local.cascade

def detect_and_crop(img_path, out_path, out_size=112, margin=0.2):
    if os.path.isfile(out_path):
        return ("cached", out_path)
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return ("failed", None)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = img_rgb.shape[:2]
    cascade = get_cascade()
    faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(20, 20))
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2] * f[3])
        mx = int(fw * margin); my = int(fh * margin)
        x0 = max(0, x - mx); y0 = max(0, y - my)
        x1 = min(w, x + fw + mx); y1 = min(h, y + fh + my)
        face = img_rgb[y0:y1, x0:x1]
        used = True
    else:
        s = min(h, w)
        face = img_rgb[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2]
        used = False
    face_resized = cv2.resize(face, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(out_path, cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR))
    # Explicitly free the big arrays so memory doesn't accumulate across threads
    del img_bgr, img_rgb, gray, face
    return ("detected" if used else "fallback", out_path)

# Build work items
print(f"[3/4] Building work items…")
work_items = []
for celeb in celebs:
    out_dir = os.path.join(CROPS_DIR, celeb)
    os.makedirs(out_dir, exist_ok=True)
    for src in celeb_to_imgs[celeb]:
        out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(src))[0] + ".jpg")
        work_items.append((celeb, src, out_path))
print(f"      Total work items: {len(work_items):,}")

print(f"[4/4] Cropping with {CROP_THREADS} threads + thread-local cascades…")
stats = defaultdict(int)
crop_paths_per_celeb = defaultdict(list)

CHUNK = 2000   # process in chunks so we can gc.collect() between them
all_pairs_sorted = work_items
processed = 0

with ThreadPoolExecutor(max_workers=CROP_THREADS) as ex:
    for chunk_start in range(0, len(all_pairs_sorted), CHUNK):
        chunk = all_pairs_sorted[chunk_start : chunk_start + CHUNK]
        futures = {ex.submit(detect_and_crop, src, out): (celeb, out) for celeb, src, out in chunk}
        for fut in tqdm(as_completed(futures), total=len(futures),
                       desc=f"chunk {chunk_start//CHUNK + 1}/{(len(all_pairs_sorted) + CHUNK - 1) // CHUNK}",
                       leave=False):
            celeb, out_path = futures[fut]
            try:
                status, path = fut.result()
            except Exception as e:
                status, path = ("error", None)
            stats[status] += 1
            if path:
                crop_paths_per_celeb[celeb].append(path)
            processed += 1
        gc.collect()

total = sum(len(v) for v in crop_paths_per_celeb.values())
print(f"\nCropping complete: {total} crops")
for k, v in stats.items():
    print(f"  {k:10s}: {v:,}")
detected = stats.get("detected", 0); fallback = stats.get("fallback", 0)
print(f"  Haar detection rate: {detected / max(1, detected + fallback):.1%}")
gc.collect()

## 6 — Pre-compute baseline embeddings

Run every cropped face through the **frozen** `mobilefacenet.tflite`. Save 512-D embeddings as `(N, 512)` float32 numpy array. From here on the actual images don't matter — training the adapter only uses these vectors.

Single-threaded TFLite (interpreter is not thread-safe), ~25 ms per image, ~5 minutes for 12k images on Kaggle CPU.

In [ ]:
import tensorflow as tf

EMBED_CACHE = f"{WORK_DIR}/baseline_embeddings.npz"

if os.path.isfile(EMBED_CACHE):
    print(f"Loading cached embeddings: {EMBED_CACHE}")
    z = np.load(EMBED_CACHE, allow_pickle=True)
    all_embeddings = z["embeddings"]
    all_labels     = z["labels"]
    all_paths      = z["paths"]
else:
    print("Pre-computing baseline embeddings via mobilefacenet.tflite…")
    it = tf.lite.Interpreter(model_path=BASELINE_TFLITE)
    it.allocate_tensors()
    inp_d = it.get_input_details()[0]
    out_d = it.get_output_details()[0]
    print(f"  Backbone input:  {inp_d['shape'].tolist()}")
    print(f"  Backbone output: {out_d['shape'].tolist()}")

    def embed_one(img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMAGE_SIZE).astype(np.float32)
        img = (img - 127.5) / 127.5
        img = img[None].astype(inp_d["dtype"])
        it.set_tensor(inp_d["index"], img)
        it.invoke()
        return it.get_tensor(out_d["index"])[0].copy()

    embeddings_list = []
    labels_list     = []
    paths_list      = []
    for celeb in tqdm(celebs, desc="per-celeb embedding"):
        label = celeb_to_idx[celeb]
        for p in crop_paths_per_celeb[celeb]:
            try:
                e = embed_one(p)
                embeddings_list.append(e)
                labels_list.append(label)
                paths_list.append(p)
            except Exception as e:
                print(f"  skip {p}: {e}")
    all_embeddings = np.stack(embeddings_list).astype(np.float32)
    all_labels     = np.array(labels_list, dtype=np.int32)
    all_paths      = np.array(paths_list, dtype=object)
    np.savez(EMBED_CACHE,
             embeddings=all_embeddings,
             labels=all_labels,
             paths=all_paths)
    print(f"\nSaved cache: {EMBED_CACHE}")

print(f"\nEmbeddings shape: {all_embeddings.shape}")
print(f"Labels    shape:  {all_labels.shape}")
print(f"Memory:           {all_embeddings.nbytes / 1024 / 1024:.1f} MB")
print(f"Embedding norm stats: min={float(np.linalg.norm(all_embeddings, axis=1).min()):.3f} mean={float(np.linalg.norm(all_embeddings, axis=1).mean()):.3f} max={float(np.linalg.norm(all_embeddings, axis=1).max()):.3f}")
print("(non-normalized; mean ~5-15 is expected for InsightFace ArcFace embeddings)")

## 7 — Train/val split (stratified per celebrity, SAME SEED as 04a/04b)

Notebook 05 reproduces this split using the same SEED to compute pair-verification metrics on identical held-out images across all candidates.

In [ ]:
import random

rng = random.Random(SEED)
train_idx, val_idx = [], []
indices_per_celeb = defaultdict(list)
for i, lab in enumerate(all_labels):
    indices_per_celeb[int(lab)].append(i)

for lab, idxs in indices_per_celeb.items():
    idxs = idxs[:]
    rng.shuffle(idxs)
    n_val = max(1, int(len(idxs) * VAL_SPLIT))
    val_idx.extend(idxs[:n_val])
    train_idx.extend(idxs[n_val:])

train_idx = np.array(train_idx, dtype=np.int64)
val_idx   = np.array(val_idx,   dtype=np.int64)

X_train = all_embeddings[train_idx]; y_train = all_labels[train_idx]
X_val   = all_embeddings[val_idx];   y_val   = all_labels[val_idx]

print(f"Train: {len(X_train)} embeddings  shape={X_train.shape}")
print(f"Val:   {len(X_val)} embeddings    shape={X_val.shape}")
print(f"Train class counts: min={int(np.bincount(y_train).min())} max={int(np.bincount(y_train).max())} mean={float(np.bincount(y_train).mean()):.1f}")
print(f"Val   class counts: min={int(np.bincount(y_val).min())}   max={int(np.bincount(y_val).max())}   mean={float(np.bincount(y_val).mean()):.1f}")

## 8 — Build the adapter network

Compact residual network: input (512) → Dense(512, no bias) → BN → tanh → Dense(512, no bias) → BN → residual-add(input) → output (512).

**Residual structure is critical:** by adding the input back at the end, the adapter learns a *delta* on the baseline embedding. At initialization the delta is near-zero, so adapted ≈ baseline. The adapter never degrades quality below baseline; it can only improve. Worst case after training: it matches baseline. Best case: it specializes for Indian faces.

In [ ]:
from tensorflow.keras import layers as L
import keras

def build_adapter(embedding_dim=512):
    inp = L.Input(shape=(embedding_dim,), name="base_embedding")
    x = L.Dense(embedding_dim, use_bias=False)(inp)
    x = L.BatchNormalization()(x)
    x = L.Activation("tanh")(x)
    x = L.Dense(embedding_dim, use_bias=False)(x)
    x = L.BatchNormalization()(x)
    out = L.Add(name="adapted_embedding")([inp, x])
    return keras.Model(inp, out, name="adapter")

adapter = build_adapter(EMBEDDING_DIM)
adapter.summary()
n_params = adapter.count_params()
print(f"\nAdapter params: {n_params:,}")
print(f"FP32 size estimate: {n_params * 4 / 1024 / 1024:.2f} MB")

## 9 — ArcFace head + training model

In [ ]:
class ArcFaceHead(tf.keras.layers.Layer):
    def __init__(self, num_classes, scale=64.0, margin=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.scale = scale
        self.margin = margin
    def build(self, input_shape):
        emb_dim = input_shape[0][-1]
        self.kernel = self.add_weight(name="kernel", shape=(emb_dim, self.num_classes),
                                      initializer="glorot_uniform", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        embeddings, labels = inputs
        eps = 1e-9
        e_n = embeddings / (keras.ops.sqrt(keras.ops.sum(keras.ops.square(embeddings), axis=1, keepdims=True)) + eps)
        k_n = self.kernel    / (keras.ops.sqrt(keras.ops.sum(keras.ops.square(self.kernel),    axis=0, keepdims=True)) + eps)
        cos_t = keras.ops.matmul(e_n, k_n)
        cos_t = keras.ops.clip(cos_t, -1.0 + 1e-7, 1.0 - 1e-7)
        sin_t = keras.ops.sqrt(1.0 - keras.ops.square(cos_t))
        cm = keras.ops.cos(self.margin); sm = keras.ops.sin(self.margin)
        cos_tm = cos_t * cm - sin_t * sm
        mask = keras.ops.one_hot(keras.ops.cast(labels, "int32"), self.num_classes)
        logits = keras.ops.where(mask > 0.5, cos_tm, cos_t)
        return logits * self.scale
    def get_config(self):
        return {**super().get_config(), "num_classes": self.num_classes, "scale": self.scale, "margin": self.margin}

emb_input   = L.Input(shape=(EMBEDDING_DIM,), name="base_embedding")
label_input = L.Input(shape=(), dtype="int32", name="label")

adapted = adapter(emb_input)
logits  = ArcFaceHead(num_classes=NUM_CLASSES, scale=ARCFACE_SCALE,
                      margin=ARCFACE_MARGIN, name="arcface")([adapted, label_input])

training_model = keras.Model([emb_input, label_input], logits, name="training_model")
training_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["sparse_categorical_accuracy"],
)
training_model.summary(line_length=120)

## 10 — Train

Up to 30 epochs (more than 04a/04b because each epoch is much faster — no images, no big backbone). EarlyStopping kills it once val_loss stops improving for 3 epochs.

In [ ]:
import json

BEST_CKPT = f"{WORK_DIR}/best_adapter.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(filepath=BEST_CKPT, monitor="val_loss",
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=2, min_lr=1e-6, verbose=1),
]

history = training_model.fit(
    x=(X_train, y_train),
    y=y_train,
    validation_data=((X_val, y_val), y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_MAX,
    callbacks=callbacks,
    verbose=2,
    shuffle=True,
)

HIST_PATH = f"{REPORTS_OUT}/mobilefacenet_adapter_training_history.json"
hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
hist_data["strategy"]      = STRATEGY
hist_data["epochs_actual"] = len(history.history.get("loss", []))
hist_data["epochs_max"]    = EPOCHS_MAX
hist_data["lr"]            = LR
hist_data["num_classes"]   = NUM_CLASSES
hist_data["best_val_loss"] = float(min(history.history.get("val_loss", [float("inf")])))
with open(HIST_PATH, "w") as f:
    json.dump(hist_data, f, indent=2)
print(f"\nHistory saved -> {HIST_PATH}")

## 11 — Export the adapter as TFLite

Just the adapter — input `[1, 512]`, output `[1, 512]`. The backbone (`mobilefacenet.tflite`) stays unchanged in the repo.

In [ ]:
# Build an inference-only adapter model (just the residual block, no ArcFace head)
inf_inp = L.Input(shape=(EMBEDDING_DIM,))
inf_out = adapter(inf_inp)
inference_model = keras.Model(inf_inp, inf_out, name="adapter_inference")

TFLITE_PATH = f"{MODELS_OUT}/{TFLITE_FILENAME}"
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
tflite_bytes = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_bytes)
print(f"Adapter TFLite saved -> {TFLITE_PATH}  ({os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB)")

# Verify shapes
it_adapter = tf.lite.Interpreter(model_path=TFLITE_PATH)
it_adapter.allocate_tensors()
ish = it_adapter.get_input_details()[0]["shape"].tolist()
osh = it_adapter.get_output_details()[0]["shape"].tolist()
print(f"Adapter input:  {ish}")
print(f"Adapter output: {osh}")
assert ish == [1, 512], f"input shape mismatch: {ish}"
assert osh == [1, 512], f"output shape mismatch: {osh}"

## 12 — Sanity check the composed (backbone + adapter) pipeline

Take a few held-out val celebs. For each, run two images through the composed pipeline (backbone TFLite → adapter TFLite), L2-normalize, and compare same-celeb vs cross-celeb cosine distance. We want:

- Same-celeb adapted distance **lower** than baseline (the adapter helps Indian face discrimination)
- Cross-celeb adapted distance **at least as high** as baseline (the adapter doesn't collapse)

In [ ]:
rng2 = random.Random(SEED + 100)

# Build per-celeb val path lists
val_paths_per_celeb = defaultdict(list)
for i in val_idx:
    val_paths_per_celeb[int(all_labels[i])].append(str(all_paths[i]))

test_celeb_idxs = rng2.sample(
    [c for c, p in val_paths_per_celeb.items() if len(p) >= 2], 3)
test_celebs = [celebs[i] for i in test_celeb_idxs]

it_back = tf.lite.Interpreter(model_path=BASELINE_TFLITE)
it_back.allocate_tensors()
back_in = it_back.get_input_details()[0]
back_out = it_back.get_output_details()[0]

ad_in = it_adapter.get_input_details()[0]
ad_out = it_adapter.get_output_details()[0]

def embed_baseline(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE).astype(np.float32)
    img = (img - 127.5) / 127.5
    it_back.set_tensor(back_in["index"], img[None].astype(back_in["dtype"]))
    it_back.invoke()
    e = it_back.get_tensor(back_out["index"])[0]
    return e

def embed_composed(img_path):
    base = embed_baseline(img_path)
    it_adapter.set_tensor(ad_in["index"], base[None].astype(ad_in["dtype"]))
    it_adapter.invoke()
    return it_adapter.get_tensor(ad_out["index"])[0]

def l2(v):
    return v / (np.linalg.norm(v) + 1e-9)

def cosd(a, b):
    return 1.0 - float(np.dot(l2(a), l2(b)))

embs_base = {c: [embed_baseline(p) for p in val_paths_per_celeb[ci][:3]]
             for c, ci in zip(test_celebs, test_celeb_idxs)}
embs_comp = {c: [embed_composed(p) for p in val_paths_per_celeb[ci][:3]]
             for c, ci in zip(test_celebs, test_celeb_idxs)}

print("Same-celeb cosine distance (lower = more similar):")
print(f"{'Celeb':<30} {'baseline':>12} {'composed':>12} {'delta':>10}")
for c in test_celebs:
    eb = embs_base[c]; ec = embs_comp[c]
    if len(eb) < 2: continue
    db = np.mean([cosd(eb[i], eb[j]) for i in range(len(eb)) for j in range(i+1, len(eb))])
    dc = np.mean([cosd(ec[i], ec[j]) for i in range(len(ec)) for j in range(i+1, len(ec))])
    delta = dc - db
    arrow = "↓ better" if delta < 0 else "↑ worse"
    print(f"  {c:<28} {db:>12.4f} {dc:>12.4f} {delta:>+10.4f}  {arrow}")

print("\nCross-celeb cosine distance (higher = more discriminative):")
print(f"{'Pair':<40} {'baseline':>12} {'composed':>12} {'delta':>10}")
for i, c1 in enumerate(test_celebs):
    for j, c2 in enumerate(test_celebs):
        if i >= j: continue
        db = cosd(embs_base[c1][0], embs_base[c2][0])
        dc = cosd(embs_comp[c1][0], embs_comp[c2][0])
        delta = dc - db
        arrow = "↑ better" if delta > 0 else "↓ worse"
        print(f"  {c1[:18]:18s} vs {c2[:18]:18s} {db:>12.4f} {dc:>12.4f} {delta:>+10.4f}  {arrow}")

## 13 — Plots + final summary

In [ ]:
import matplotlib.pyplot as plt

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, ylabel) in zip(axes, [("loss", "Loss"), ("sparse_categorical_accuracy", "Accuracy")]):
    if key in h and f"val_{key}" in h:
        ax.plot(h[key], label=f"train {key}")
        ax.plot(h[f"val_{key}"], label=f"val {key}")
        ax.set_xlabel("epoch"); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle(f"MobileFaceNet ADAPTER (frozen backbone + Keras head) on Bollywood Faces\n(best val_loss={hist_data['best_val_loss']:.4f})")
fig.tight_layout()
PLOT_PATH = f"{REPORTS_OUT}/mobilefacenet_adapter_training_curves.png"
fig.savefig(PLOT_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"Plot saved: {PLOT_PATH}")

print("\n" + "=" * 78)
print(f"Notebook 04c — FINAL SUMMARY ({STRATEGY})")
print("=" * 78)
print(f"Total Bollywood images cropped:  {len(all_embeddings):,}")
print(f"Train embeddings:                {len(X_train):,}")
print(f"Val embeddings:                  {len(X_val):,}")
print(f"Adapter params:                  {adapter.count_params():,}")
print(f"Epochs ran:                      {hist_data['epochs_actual']} of {EPOCHS_MAX} max")
print(f"Final train acc:                 {h['sparse_categorical_accuracy'][-1]:.4f}")
print(f"Final val   acc:                 {h['val_sparse_categorical_accuracy'][-1]:.4f}")
print(f"Best  val   loss:                {hist_data['best_val_loss']:.4f}")
print()
print(f"Backbone (unchanged):            {BASELINE_TFLITE}  ({os.path.getsize(BASELINE_TFLITE)/1024/1024:.2f} MB)")
print(f"Adapter TFLite (new):            {TFLITE_PATH}  ({os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB)")
print(f"Composed bundle size:            {(os.path.getsize(BASELINE_TFLITE) + os.path.getsize(TFLITE_PATH))/1024/1024:.2f} MB")
print(f"Keras checkpoint:                {BEST_CKPT}  ({os.path.getsize(BEST_CKPT)/1024/1024:.2f} MB)")
print(f"History JSON:                    {HIST_PATH}")
print(f"Curves PNG:                      {PLOT_PATH}")

## What to send back to Claude

Paste literal output of these cells (in order):

1. Cell 7 (dataset discovery)
2. Cell 9 (per-celeb gathering)
3. Cell 11 (cropping with threading — should be MUCH faster than 04a/04b)
4. Cell 13 (baseline embedding pre-compute — embedding shape and norm stats)
5. Cell 15 (train/val split sizes)
6. Cell 17 (adapter architecture summary + params)
7. Cell 19 (training model summary)
8. Cell 21 (per-epoch training output)
9. Cell 23 (adapter TFLite size + shape check)
10. Cell 25 (sanity check — same-celeb and cross-celeb cosine distance: baseline vs composed)
11. Cell 27 (final summary)

## After it runs — what to do on the laptop

Download to `kaggle_downloads/04c_mobilefacenet_adapter/`:

- `/kaggle/working/models/mobilefacenet_adapter.tflite`
- `/kaggle/working/best_adapter.keras` (for Notebook 05's INT8 PTQ on the adapter)
- `/kaggle/working/reports/mobilefacenet_adapter_training_history.json`
- `/kaggle/working/reports/mobilefacenet_adapter_training_curves.png`

Then upload them into the **`mobilefacenet-candidates`** Kaggle Dataset alongside 04b's files, under a new `04c/` subfolder. Notebook 05 will then evaluate baseline + 04b + 04c (and 04a if you got it working) on Bollywood pair verification and ship the EER winner.

## How the mobile side uses this

Two `.tflite` files load instead of one. JSI processor changes:

```
image (112x112x3, [-1,1])
    │
    ▼
mobilefacenet.tflite           (frozen backbone)
    │
    ▼
raw 512-D embedding
    │
    ▼
mobilefacenet_adapter.tflite   (Indian-face tuned)
    │
    ▼
adapted 512-D embedding
    │
    ▼
L2-normalize                    (mobile side, per the contract)
    │
    ▼
cosine distance vs stored template
```

Total inference: ~2 ms extra for the adapter pass on top of backbone latency. Negligible.